## ENV SETUP

1. Install uv (or do it you're own way)
2. Run `uv sync`
3. Run `source .venv/bin/activate`

You're good to go.

# Instructions

The Task : Create the best CadQuery code generator model. 

1. Load the dataset (147K pairs of Images/CadQuery code).
2. Create a baseline model and evaluate it with the given metrics.
3. Enhance by any manner the baseline model and evaluate it again.
4. Explain you choices and possible bottlenecks. 
5. Show what enhancements you would have done if you had more time.

You can do *WHATEVER* you want, be creative, result is not what matters the most. 
Creating new model architectures, reusing ones you used in the past, fine-tuning, etc...

If you are GPU poor, there are solutions. Absolute value is not what matters, relative value between baseline and enhanced model is what matters.

In [ ]:
from datasets import load_dataset
ds = load_dataset("CADCODER/GenCAD-Code", num_proc=16, split=["train", "test"], cache_dir="/Volumes/BIG-DATA/HUGGINGFACE_CACHE")

## Evaluation Metrics

1. Valid Syntax Rate metric assess the validity of the code by executing and checking if error are returned.
2. Best IOU assess the similarity between the meshes generated by the code.

In [ ]:
from metrics.valid_syntax_rate import evaluate_syntax_rate_simple
from metrics.best_iou import get_iou_best

In [ ]:
## Example usage of the metrics
sample_code = """
height = 60.0
width = 80.0
thickness = 10.0
diameter = 22.0

# make the base
result = (
    cq.Workplane("XY")
    .box(height, width, thickness)
)
"""

sample_code_2 = """
 height = 60.0
 width = 80.0
 thickness = 10.0
 diameter = 22.0
 padding = 12.0

 # make the base
 result = (
     cq.Workplane("XY")
     .box(height, width, thickness)
     .faces(">Z")
     .workplane()
     .hole(diameter)
     .faces(">Z")
     .workplane()
     .rect(height - padding, width - padding, forConstruction=True)
     .vertices()
     .cboreHole(2.4, 4.4, 2.1)
 )
"""
codes = {
    "sample_code": sample_code,
    "sample_code_2": sample_code_2,
}
vsr = evaluate_syntax_rate_simple(codes)
print("Valid Syntax Rate:", vsr)
iou = get_iou_best(sample_code, sample_code_2)
print("IOU:", iou)

## Have Fun

# ***Joseph***

In [ ]:
from datasets import load_dataset

ds = load_dataset("CADCODER/GenCAD-Code", split=["train", "test"])
train_ds, test_ds = ds[0], ds[1]



In [ ]:
train_ds[0]

In [ ]:
print("Train size:", len(train_ds))

In [ ]:
import re

def tokenize_cadquery(code: str) -> list[str]:
    """
    Tokenizes CadQuery code into meaningful units: identifiers, numbers, symbols, dots, parentheses, etc.
    """
    # Remove comments
    code = re.sub(r"#.*", "", code)
    
    # Pattern to capture words, floats, symbols, parentheses, etc.
    token_pattern = r"""[\w\.]+|[=\(\)\[\]\{\},\.\"]|\d+\.\d+|\d+"""
    tokens = re.findall(token_pattern, code)
    
    return tokens


In [ ]:
example = train_ds[0]["cadquery"]
tokens = tokenize_cadquery(example)
print(tokens[:100])  # just the first 50 tokens
print("Token count:", len(tokens))


In [ ]:
from collections import Counter
from tqdm import tqdm

token_counter = Counter()

# Loop through a subset (or all) of train_ds to build vocab
for sample in tqdm(train_ds, desc="Tokenizing"):
    tokens = tokenize_cadquery(sample["cadquery"])
    token_counter.update(tokens)

# Define vocabulary (limit to top 10K if needed)
MAX_VOCAB_SIZE = 10000

# Special tokens
PAD = "<pad>"
SOS = "<sos>"
EOS = "<eos>"
UNK = "<unk>"

most_common_tokens = [token for token, _ in token_counter.most_common(MAX_VOCAB_SIZE - 4)]
vocab_tokens = [PAD, SOS, EOS, UNK] + most_common_tokens

token2id = {token: idx for idx, token in enumerate(vocab_tokens)}
id2token = {idx: token for token, idx in token2id.items()}


In [ ]:
import torch
from torch.utils.data import Dataset
from torchvision import transforms
from PIL import Image

class CadQueryDataset(Dataset):
    def __init__(self, hf_dataset, token2id, max_length=512):
        self.dataset = hf_dataset
        self.token2id = token2id
        self.max_length = max_length
        
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
        ])

    def encode_tokens(self, tokens):
        ids = [self.token2id.get("<sos>")]
        for token in tokens:
            ids.append(self.token2id.get(token, self.token2id["<unk>"]))
        ids.append(self.token2id.get("<eos>"))

        # Pad or truncate
        if len(ids) < self.max_length:
            ids += [self.token2id["<pad>"]] * (self.max_length - len(ids))
        else:
            ids = ids[:self.max_length]

        return torch.tensor(ids, dtype=torch.long)

    def __getitem__(self, idx):
        sample = self.dataset[idx]

        # Load and transform image
        image = sample["image"]
        if isinstance(image, str):
            image = Image.open(image)
        image = self.transform(image)

        tokens = tokenize_cadquery(sample["cadquery"])
        token_ids = self.encode_tokens(tokens)

       
        input_ids = token_ids[:-1]
        target_ids = token_ids[1:]

        return image, input_ids, target_ids

    def __len__(self):
        return len(self.dataset)


In [ ]:
train_data = CadQueryDataset(train_ds, token2id, max_length=512)
image, input_ids, target_ids = train_data[0]

print("Image shape:", image.shape)            
print("Input token ids:", input_ids[:100])    
print("Target token ids:", target_ids[:100])  

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

class CadQueryModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=256, hidden_dim=512, max_length=512):
        super().__init__()
        self.vocab_size = vocab_size
        self.max_length = max_length

        #  ResNet18 
        base_resnet = models.resnet18(pretrained=True)
        modules = list(base_resnet.children())[:-1] 
        self.image_encoder = nn.Sequential(*modules)
        self.feature_dim = base_resnet.fc.in_features  

       #embedding
        self.embedding = nn.Embedding(vocab_size, embed_dim)

        # GRU
        self.gru = nn.GRU(
            input_size=embed_dim + self.feature_dim,  
            hidden_size=hidden_dim,
            batch_first=True
        )

        
        self.output_layer = nn.Linear(hidden_dim, vocab_size)

    def forward(self, images, input_token_ids):
        batch_size, seq_len = input_token_ids.size()

       
        img_features = self.image_encoder(images).view(batch_size, 1, self.feature_dim)
        img_features = img_features.expand(-1, seq_len, -1)  
        token_embeds = self.embedding(input_token_ids)
        decoder_input = torch.cat([token_embeds, img_features], dim=-1)  
        outputs, _ = self.gru(decoder_input)  
        logits = self.output_layer(outputs)

        return logits


In [ ]:
import torch
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence
import torch.nn as nn
import torch.optim as optim


MAX_SAMPLES = 2000          # Subset for fast training
BATCH_SIZE = 2              
EPOCHS = 2                
MAX_LENGTH = 256           
DEVICE = torch.device("cpu")  
PAD_ID = token2id["<pad>"]


train_ds_small = train_ds.select(range(MAX_SAMPLES))


train_loader = DataLoader(
    CadQueryDataset(train_ds_small, token2id, max_length=MAX_LENGTH),
    batch_size=BATCH_SIZE,
    shuffle=True
)

model = CadQueryModel(vocab_size=len(token2id)).to(DEVICE)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)
optimizer = optim.Adam(model.parameters(), lr=1e-4)


for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for i, batch in enumerate(train_loader):
        images, input_ids, target_ids = batch
        images = images.to(DEVICE)
        input_ids = input_ids.to(DEVICE)
        target_ids = target_ids.to(DEVICE)

        optimizer.zero_grad()
        logits = model(images, input_ids)

        
        logits = logits.view(-1, logits.size(-1))
        target_ids = target_ids.view(-1)

        loss = criterion(logits, target_ids)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # progress
        if i % 10 == 0:
            print(f"[Epoch {epoch+1}] Batch {i}/{len(train_loader)} - Loss: {loss.item():.4f}")

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1} | Avg Loss: {avg_loss:.4f}")

torch.save(model.state_dict(), "cadquery_baseline.pth")


In [ ]:
import torch
from torchvision import transforms
from PIL import Image
from collections import Counter
import re


def tokens_to_code(tokens):
    """ Converts a list of tokens to syntactically correct, clean Python code for CadQuery.
    """
    import re

    lines = []
    current_line = ""
    paren_stack = [] 

    i = 0
    while i < len(tokens):
        
        if (
            i + 2 < len(tokens)
            and re.fullmatch(r"\d+(\.\d+)?e", tokens[i])
            and tokens[i + 1] in ["-", "+"]
            and re.fullmatch(r"\d+", tokens[i + 2])
        ):
            token = tokens[i] + tokens[i + 1] + tokens[i + 2]
            i += 3
        elif (
            i + 1 < len(tokens)
            and tokens[i].endswith("e")
            and re.fullmatch(r"\d+", tokens[i + 1])
        ):
            token = tokens[i] + tokens[i + 1]
            i += 2
        else:
            token = tokens[i]
            i += 1

       
        if token == "import" and i + 2 < len(tokens) and tokens[i] == "cadquery" and tokens[i + 1] == "as" and tokens[i + 2] == "cq":
            lines.append("import cadquery as cq")
            i += 3
            continue

    
        if token in ["(", "[", "{"]:
            current_line += token
            paren_stack.append(token)
            continue
        elif token in [")", "]", "}"]:
            if paren_stack:
                paren_stack.pop()
            current_line += token
            continue
        elif token == "=":
            current_line += " ="
            continue
        elif token in [".", ","]:
            current_line += token if token != "," else ", "
            continue


        if len(current_line) > 0 and current_line[-1] in [")", "]", "}"] and not token.startswith("."):
           
            if ".moveTo" in current_line and ".close()" in current_line and ".lineTo" not in current_line and ".circle" not in current_line:
                current_line = ""
                continue
            lines.append(current_line.strip())
            current_line = ""



        if current_line and not current_line.endswith(("=", "(", ".", ",")):
            current_line += " "

        current_line += token

    if current_line.strip() and not re.fullmatch(r"[a-zA-Z_][a-zA-Z0-9_]*\s*=", current_line.strip()):
        lines.append(current_line.strip())


    closing_map = {"(": ")", "[": "]", "{": "}"}
    for open_token in reversed(paren_stack):
        lines[-1] += closing_map[open_token]

    code = "\n".join(lines)


    if "extrude" not in code and "Workplane" in code:
        code += "\nsolid = wp_sketch0.extrude(1.0)"

    return code





def generate_code(model, image, token2id, id2token, max_length=512, device="cpu"):
    """
    Runs inference to generate CadQuery code from an image using a trained model.
    """
    model.eval()


    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
    ])
    image_tensor = transform(image).unsqueeze(0).to(device)

    # Token IDs
    sos_token = token2id["<sos>"]
    eos_token = token2id["<eos>"]
    pad_token = token2id["<pad>"]

    input_ids = [sos_token]

    for _ in range(max_length):
        input_tensor = torch.tensor(input_ids, dtype=torch.long).unsqueeze(0).to(device)
        with torch.no_grad():
            logits = model(image_tensor, input_tensor)
            next_token_logits = logits[0, -1, :]
            next_token = torch.argmax(next_token_logits).item()

        input_ids.append(next_token)

        if next_token == eos_token:
            break


    tokens = [id2token[token_id] for token_id in input_ids if token_id not in [sos_token, eos_token, pad_token]]


    code = tokens_to_code(tokens)
    print(" Decoded tokens (raw):", tokens)
    print(" Number of predicted tokens:", len(tokens))
    print(" Token IDs:", input_ids)


    return code


In [ ]:
model.load_state_dict(torch.load("cadquery_baseline.pth", map_location="cpu"))
model.to("cpu")
model.eval()


generated_codes = {}
for i in range(5):
    test_sample = test_ds[i]
    image = test_sample["image"]
    true_code = test_sample["cadquery"]

    pred_code = generate_code(model, image, token2id, id2token, device="cpu")
    print(f"\n Generated code (raw):\n{pred_code}\n")
    try:
        exec(pred_code)
        print(" This code is valid Python!")
    except Exception as e:
        print(" Invalid Python syntax:\n", e)

    generated_codes[f"sample_{i}"] = pred_code

    print(f"\n--- Sample {i} ---")
    print("Predicted:")
    print(pred_code)
    print("\nGround Truth:")
    print(true_code)


In [ ]:
from metrics.valid_syntax_rate import evaluate_syntax_rate_simple
from metrics.best_iou import get_iou_best


syntax_score = evaluate_syntax_rate_simple(generated_codes)
print("\n Valid Syntax Rate:", syntax_score)


iou = get_iou_best(generated_codes["sample_0"], test_ds[0]["cadquery"])
print(" IOU:", iou)


Our baseline model achieves a high valid syntax rate, showing that it learns CadQuery structure. However, the semantic IOU remains low (1%), which indicates poor alignment with the true code. This highlights the need for more training

In [ ]:
import torch.nn.functional as F


import torch
from torchvision import transforms
from PIL import Image
from collections import Counter
import re


def tokens_to_code(tokens):
    """ Converts a list of tokens to syntactically correct, clean Python code for CadQuery.
    """
    import re

    lines = []
    current_line = ""
    paren_stack = [] 

    i = 0
    while i < len(tokens):
        
        if (
            i + 2 < len(tokens)
            and re.fullmatch(r"\d+(\.\d+)?e", tokens[i])
            and tokens[i + 1] in ["-", "+"]
            and re.fullmatch(r"\d+", tokens[i + 2])
        ):
            token = tokens[i] + tokens[i + 1] + tokens[i + 2]
            i += 3
        elif (
            i + 1 < len(tokens)
            and tokens[i].endswith("e")
            and re.fullmatch(r"\d+", tokens[i + 1])
        ):
            token = tokens[i] + tokens[i + 1]
            i += 2
        else:
            token = tokens[i]
            i += 1

       
        if token == "import" and i + 2 < len(tokens) and tokens[i] == "cadquery" and tokens[i + 1] == "as" and tokens[i + 2] == "cq":
            lines.append("import cadquery as cq")
            i += 3
            continue

    
        if token in ["(", "[", "{"]:
            current_line += token
            paren_stack.append(token)
            continue
        elif token in [")", "]", "}"]:
            if paren_stack:
                paren_stack.pop()
            current_line += token
            continue
        elif token == "=":
            current_line += " ="
            continue
        elif token in [".", ","]:
            current_line += token if token != "," else ", "
            continue


        if len(current_line) > 0 and current_line[-1] in [")", "]", "}"] and not token.startswith("."):
           
            if ".moveTo" in current_line and ".close()" in current_line and ".lineTo" not in current_line and ".circle" not in current_line:
                current_line = ""
                continue
            lines.append(current_line.strip())
            current_line = ""



        if current_line and not current_line.endswith(("=", "(", ".", ",")):
            current_line += " "

        current_line += token

    if current_line.strip() and not re.fullmatch(r"[a-zA-Z_][a-zA-Z0-9_]*\s*=", current_line.strip()):
        lines.append(current_line.strip())


    closing_map = {"(": ")", "[": "]", "{": "}"}
    for open_token in reversed(paren_stack):
        lines[-1] += closing_map[open_token]

    code = "\n".join(lines)


    if "extrude" not in code and "Workplane" in code:
        code += "\nsolid = wp_sketch0.extrude(1.0)"

    return code





def generate_code(model, image, token2id, id2token, max_length=512, device="cpu", top_k=10):

    """
    Runs inference to generate CadQuery code from an image using a trained model.
    """
    model.eval()


    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
    ])
    image_tensor = transform(image).unsqueeze(0).to(device)

    # Token IDs
    sos_token = token2id["<sos>"]
    eos_token = token2id["<eos>"]
    pad_token = token2id["<pad>"]

    input_ids = [sos_token]

    for _ in range(max_length):
        input_tensor = torch.tensor(input_ids, dtype=torch.long).unsqueeze(0).to(device)
        with torch.no_grad():
            logits = model(image_tensor, input_tensor)
            next_token_logits = logits[0, -1, :]
            top_k = min(top_k, next_token_logits.size(-1)) 

            probs = F.softmax(next_token_logits, dim=-1) 
            top_k_probs, top_k_indices = torch.topk(probs, k=top_k)

            next_token = top_k_indices[torch.multinomial(top_k_probs, 1).item()].item()


        input_ids.append(next_token)

        if next_token == eos_token:
            break


    tokens = [id2token[token_id] for token_id in input_ids if token_id not in [sos_token, eos_token, pad_token]]


    code = tokens_to_code(tokens)
    print(" Decoded tokens (raw):", tokens)
    print(" Number of predicted tokens:", len(tokens))
    print(" Token IDs:", input_ids)


    return code


In [ ]:
model.load_state_dict(torch.load("cadquery_baseline.pth", map_location="cpu"))
model.to("cpu")
model.eval()


generated_codes = {}
for i in range(5):
    test_sample = test_ds[i]
    image = test_sample["image"]
    true_code = test_sample["cadquery"]

    pred_code = generate_code(model, image, token2id, id2token, device="cpu")
    print(f"\n Generated code (raw):\n{pred_code}\n")
    try:
        exec(pred_code)
        print(" This code is valid Python!")
    except Exception as e:
        print(" Invalid Python syntax:\n", e)

    generated_codes[f"sample_{i}"] = pred_code

    print(f"\n--- Sample {i} ---")
    print("Predicted:")
    print(pred_code)
    print("\nGround Truth:")
    print(true_code)


In [ ]:
from metrics.valid_syntax_rate import evaluate_syntax_rate_simple
from metrics.best_iou import get_iou_best


syntax_score = evaluate_syntax_rate_simple(generated_codes)
print("\n Valid Syntax Rate:", syntax_score)


iou = get_iou_best(generated_codes["sample_0"], test_ds[0]["cadquery"])
print(" IOU:", iou)
